In [1]:
def load_folder_pickles_peptide(method_folder, peptide_name):
# method_data    Load pickle files from a specific peptide subfolder within a method folder.    Only loads the first 5 pickle files from the peptide subfolder.        Args:
#         method_folder (str): Name of the method folder (e.g., "bo", "cbas", etc.)
#         peptide_name (str): Name of the peptide subfolder (e.g., "KYCRRFRWLTFRWL")
    
#     Returns:
#         dict: Dictionary with structure {seed_name: data} for the specific peptide
    
    import pickle
    import os
    from pathlib import Path
    
    peptide_data = {}
    peptide_path = Path(method_folder) / peptide_name
    
    if not peptide_path.exists():
        print(f"Path {peptide_path} does not exist!")
        return peptide_data
    
    print(f"Loading data from {method_folder}/{peptide_name}/ folder...")
    
    # Load only the first 5 pickle files in this peptide folder
    pkl_files = sorted(list(peptide_path.glob("*.pkl")))[:5]
    
    for pkl_file in pkl_files:
        try:
            with open(pkl_file, 'rb') as f:
                data = pickle.load(f)
                seed_name = pkl_file.stem  # filename without extension
                peptide_data[seed_name] = data
                print(f"  Loaded {pkl_file}")
        except Exception as e:
            print(f"  Error loading {pkl_file}: {e}")
    
    print(f"\nLoaded {len(peptide_data)} runs for peptide {peptide_name}")
    for seed_name in peptide_data.keys():
        print(f"  {seed_name}")
    
    return peptide_data

In [2]:
# Test the peptide-specific loading function
print("🧪 Testing peptide-specific pickle loading...")

# Load data for a specific peptide from the BO method
peptide_name = "KYCRRFRWLTFRWL"
method_folder = "bo"

peptide_data = load_folder_pickles_peptide(method_folder, peptide_name)

if peptide_data:
    print(f"\n📊 Successfully loaded data for {peptide_name}")
    print(f"Seeds loaded: {list(peptide_data.keys())}")
    
    # Show structure of first seed
    first_seed = list(peptide_data.keys())[0]
    first_seed_data = peptide_data[first_seed]
    print(f"\nStructure of {first_seed}:")
    print(f"  Number of steps: {len(first_seed_data)}")
    print(f"  First few steps: {list(first_seed_data.keys())[:5]}")
    
    # Show sample step data
    sample_step = list(first_seed_data.keys())[5]
    step_data = first_seed_data[sample_step]
    print(f"\nSample step {sample_step}:")
    if isinstance(step_data, dict):
        print(f"  Keys: {list(step_data.keys())}")
        if 'scores' in step_data:
            scores = step_data['scores']
            print(f"  Scores: {scores[:3] if isinstance(scores, list) else scores}")
        if 'sequences' in step_data:
            sequences = step_data['sequences']
            print(f"  Sequences: {sequences[:2] if isinstance(sequences, list) else sequences}")
else:
    print(f"❌ Failed to load data for {peptide_name}")

print(f"\n🎯 Use this function to load specific peptides:")
print(f"   peptide_data = load_folder_pickles_peptide('method_name', 'PEPTIDE_SEQUENCE')")
print(f"   Available peptides: {peptides}")

🧪 Testing peptide-specific pickle loading...
Loading data from bo/KYCRRFRWLTFRWL/ folder...
  Loaded bo\KYCRRFRWLTFRWL\seed_1.pkl
  Loaded bo\KYCRRFRWLTFRWL\seed_10.pkl
  Loaded bo\KYCRRFRWLTFRWL\seed_2.pkl
  Loaded bo\KYCRRFRWLTFRWL\seed_3.pkl
  Loaded bo\KYCRRFRWLTFRWL\seed_4.pkl

Loaded 5 runs for peptide KYCRRFRWLTFRWL
  seed_1
  seed_10
  seed_2
  seed_3
  seed_4

📊 Successfully loaded data for KYCRRFRWLTFRWL
Seeds loaded: ['seed_1', 'seed_10', 'seed_2', 'seed_3', 'seed_4']

Structure of seed_1:
  Number of steps: 468
  First few steps: [0, 1, 2, 3, 4]

Sample step 5:
  Keys: ['sequences', 'scores']
  Scores: [-13.347392082214355, -13.348187446594238, -13.349786758422852]
  Sequences: ['KDHRGNAMLTEQAD', 'KDHRGDAMLTRVPD']

🎯 Use this function to load specific peptides:
   peptide_data = load_folder_pickles_peptide('method_name', 'PEPTIDE_SEQUENCE')


NameError: name 'peptides' is not defined

In [3]:
peptide_data

{'seed_1': {0: {'sequences': ['KYCRRFRWLTFRWL'], 'scores': [-1.8769375]},
  1: {'sequences': ['KTCRGFRWLTNRWL', 'KTCRARRWCTNRWL', 'KTCRGFRWLTFRWL'],
   'scores': [-3.1119301319122314, -3.5256264209747314, -2.999396562576294]},
  2: {'sequences': ['KTQRGMAWLTFRWK', 'KTARGFEWLTNRWL', 'KTQRGFAWLTFRWK'],
   'scores': [-5.177945137023926, -7.416021823883057, -6.714558124542236]},
  3: {'sequences': ['KAQRGDAWLTRRDD', 'KAQRGDAWLTRRFD', 'KTQRGDAWLTRRWD'],
   'scores': [-13.331618309020996, -13.22002124786377, -12.136636734008789]},
  4: {'sequences': ['KDHRGDAMLTRRWD', 'KTQRGDAMLTRRWD', 'KDHGGDAMLTRRWD'],
   'scores': [-13.296060562133789, -13.339471817016602, -13.362728118896484]},
  5: {'sequences': ['KDHRGNAMLTEQAD', 'KDHRGDAMLTRVPD', 'KDHRGDAMLTERAD'],
   'scores': [-13.347392082214355, -13.348187446594238, -13.349786758422852]},
  6: {'sequences': ['KSHRGNSYLTEQAD', 'KSHRGNSYRTEQWD', 'KDHRGNAYLTEQAD'],
   'scores': [-13.343862533569336, -13.319923400878906, -13.345185279846191]},
  7: {'

In [4]:
def get_vectors_for_each_seed_csv(peptide_data, method_name):
    import pandas as pd
    from pathlib import Path
    
    ##create number of keys(so for each seed) scores, sequences vectors. As they are stored in steps having multiple scores and sequences, you have to open them and append to final vector
    scores_dict = {}
    sequences_dict = {}
    vectors_dict = {}

    for seed_name, seed_data in peptide_data.items():
        scores_list = []
        sequences_list = []
        vectors_list = []
        
        for step, step_data in seed_data.items():
            if 'scores' in step_data:
                scores_list.extend(step_data['scores'])
            if 'sequences' in step_data:
                sequences_list.extend(step_data['sequences'])
            if 'vectors' in step_data:
                vectors_list.extend(step_data['vectors'])
        
        scores_dict[seed_name] = scores_list
        sequences_dict[seed_name] = sequences_list
        vectors_dict[seed_name] = vectors_list

    ##stack scores with sequences for each seed and save in csv_outputs/method_name/peptide_name_seedname.csv
    output_folder = Path("csv_outputs") / method_name
    output_folder.mkdir(parents=True, exist_ok=True)
    for seed_name in peptide_data.keys():
        df = pd.DataFrame({
            'Sequence': sequences_dict[seed_name],
            'Score': scores_dict[seed_name]
        })
        output_file = output_folder / f"{method_name}_{seed_name}.csv"
        df.to_csv(output_file, index=False)
        print(f"Saved CSV for {seed_name} at {output_file}")
    return scores_dict, sequences_dict, vectors_dict



    

In [ ]:
peptides = [ "KYCRRFRWLTFRWL","KFRNRHRWKFKLIFRN",  "KKYWLIRKWIRLWFLT","FLYKWWIRIGRLKL", "KTLKIIRLLF","RMARNLVRYVQGLKKKKVI"]

names = [ "KY14","KF16", "KK16","FL14", "mammuthusin-3", "hydrodamin-2" ]

methods = ["bo", "cbas", "adalead", 'dynappo', 'gfn_al', 'gfn_al_cs', 'cmaes','pex']

In [6]:
for peptide, name in zip(peptides, names):
    for method in methods:
        print(f"\nProcessing method: {method}, peptide: {peptide} ({name})")
        peptide_data = load_folder_pickles_peptide(method, peptide)
        if peptide_data:
            scores_dict, sequences_dict, vectors_dict = get_vectors_for_each_seed_csv(peptide_data, f"{method}_{name}")
        else:
            print(f"No data found for method: {method}, peptide: {peptide} ({name})")


Processing method: bo, peptide: KYCRRFRWLTFRWL (KY14)
Loading data from bo/KYCRRFRWLTFRWL/ folder...
  Loaded bo\KYCRRFRWLTFRWL\seed_1.pkl
  Loaded bo\KYCRRFRWLTFRWL\seed_10.pkl
  Loaded bo\KYCRRFRWLTFRWL\seed_2.pkl
  Loaded bo\KYCRRFRWLTFRWL\seed_3.pkl
  Loaded bo\KYCRRFRWLTFRWL\seed_4.pkl

Loaded 5 runs for peptide KYCRRFRWLTFRWL
  seed_1
  seed_10
  seed_2
  seed_3
  seed_4
Saved CSV for seed_1 at csv_outputs\bo_KY14\bo_KY14_seed_1.csv
Saved CSV for seed_10 at csv_outputs\bo_KY14\bo_KY14_seed_10.csv
Saved CSV for seed_2 at csv_outputs\bo_KY14\bo_KY14_seed_2.csv
Saved CSV for seed_3 at csv_outputs\bo_KY14\bo_KY14_seed_3.csv
Saved CSV for seed_4 at csv_outputs\bo_KY14\bo_KY14_seed_4.csv

Processing method: cbas, peptide: KYCRRFRWLTFRWL (KY14)
Loading data from cbas/KYCRRFRWLTFRWL/ folder...
  Loaded cbas\KYCRRFRWLTFRWL\seed_1.pkl
  Loaded cbas\KYCRRFRWLTFRWL\seed_10.pkl
  Loaded cbas\KYCRRFRWLTFRWL\seed_2.pkl
  Loaded cbas\KYCRRFRWLTFRWL\seed_3.pkl
  Loaded cbas\KYCRRFRWLTFRWL\seed_4

In [7]:
import pandas as pd
from pathlib import Path

def max_scores_for_sequence(folder_path):
    """
    Takes a folder path and reads all CSV files from it.
    Extracts maximum scores from 'Score' or 'score' column in each CSV file.
    
    Args:
        folder_path (str or Path): Path to folder containing CSV files
    
    Returns:
        tuple: (initial_score, max_scores_list, mean, std)
    """
    
    folder_path = Path(folder_path)
    
    if not folder_path.exists():
        raise FileNotFoundError(f"Folder does not exist: {folder_path}")
    
    result_max_scores = []
    initial_score = None
    
    # Get all CSV files in the folder
    csv_files = sorted(folder_path.glob("*.csv"))
    
    if len(csv_files) == 0:
        print(f"No CSV files found in: {folder_path}")
        return None, [], None, None
    
    print(f"Found {len(csv_files)} CSV files in {folder_path}")
    
    for csv_file in csv_files:
        try:
            df = pd.read_csv(csv_file)
            
            # Check for score column (case insensitive)
            score_col = None
            for col in df.columns:
                if col.lower() == 'score':
                    score_col = col
                    break
            
            if score_col is None:
                print(f"No 'score' column found in {csv_file}")
                continue
            
            # Get max score and initial score
            max_score = -df[score_col].max()  # Negative because higher score is better
            result_max_scores.append(max_score)
            
            # Get initial score from first CSV file
            if initial_score is None:
                initial_score = -df[score_col].iloc[0]
            
            print(f"  {csv_file.name}: max_score = {max_score:.3f}")
            
        except Exception as e:
            print(f"Error reading {csv_file}: {e}")
    
    # Calculate statistics
    if result_max_scores:
        mean = np.mean(result_max_scores)
        sd = np.std(result_max_scores)
        print(f"\nSummary: {len(result_max_scores)} files processed")
        print(f"Mean max score: {mean:.3f} ± {sd:.3f}")
    else:
        mean = None
        sd = None
    
    return initial_score, result_max_scores, mean, sd


In [8]:
import numpy as np
import os

In [9]:
matrix = {}

for method in methods:
    row = {}
    for name in names:
        folder = f"csv_outputs/{method}_{name}"
        if not os.path.exists(folder):
            row[name] = ""
            continue
        
        initial_score, result_max_scores, mean, sd = max_scores_for_sequence(folder)
        
        # zapis jako "mean ± sd"
        row[name] = f"{mean:.2f} ± {sd:.2f}"
    
    matrix[method] = row

df = pd.DataFrame(matrix).T  # metody jako wiersze
print(df)

Found 5 CSV files in csv_outputs\bo_KY14
  bo_KY14_seed_1.csv: max_score = 0.755
  bo_KY14_seed_10.csv: max_score = 0.723
  bo_KY14_seed_2.csv: max_score = 0.539
  bo_KY14_seed_3.csv: max_score = 0.714
  bo_KY14_seed_4.csv: max_score = 0.838

Summary: 5 files processed
Mean max score: 0.714 ± 0.098
Found 5 CSV files in csv_outputs\bo_KK16
  bo_KK16_seed_1.csv: max_score = 0.622
  bo_KK16_seed_10.csv: max_score = 0.726
  bo_KK16_seed_2.csv: max_score = 0.570
  bo_KK16_seed_3.csv: max_score = 0.575
  bo_KK16_seed_4.csv: max_score = 0.539

Summary: 5 files processed
Mean max score: 0.606 ± 0.066
Found 5 CSV files in csv_outputs\bo_FL14
  bo_FL14_seed_1.csv: max_score = 0.662
  bo_FL14_seed_10.csv: max_score = 0.749
  bo_FL14_seed_2.csv: max_score = 0.755
  bo_FL14_seed_3.csv: max_score = 0.796
  bo_FL14_seed_4.csv: max_score = 0.777

Summary: 5 files processed
Mean max score: 0.748 ± 0.046
Found 5 CSV files in csv_outputs\bo_mammuthusin-3
  bo_mammuthusin-3_seed_1.csv: max_score = 0.987
 

In [10]:
max_scores_for_sequence("csv_outputs/gfn_al_cs_FL14")

Found 5 CSV files in csv_outputs\gfn_al_cs_FL14
  gfn_al_cs_FL14_seed_1.csv: max_score = -4.653
  gfn_al_cs_FL14_seed_10.csv: max_score = -8.728
  gfn_al_cs_FL14_seed_2.csv: max_score = -8.275
  gfn_al_cs_FL14_seed_3.csv: max_score = -8.650
  gfn_al_cs_FL14_seed_4.csv: max_score = 1.168

Summary: 5 files processed
Mean max score: -5.828 ± 3.813


(1.6251498,
 [-4.653388500213623,
  -8.72793960571289,
  -8.274609565734863,
  -8.650304794311523,
  1.1683671474456787],
 -5.827575063705444,
 3.8128681322687963)

In [11]:
def create_results_table(methods, peptides, names, base_dir="csv_outputs"):
    """
    Create a comprehensive results table with methods as rows and peptides as columns.
    Includes an initial values row and mean ± std for each method-peptide combination.
    
    Args:
        methods (list): List of optimization methods
        peptides (list): List of peptide sequences  
        names (list): List of peptide names corresponding to sequences
        base_dir (str): Base directory containing the CSV folders
    
    Returns:
        pandas.DataFrame: Results table
    """
    
    import pandas as pd
    from pathlib import Path
    
    print("🎯 Creating comprehensive results table...")
    print("="*60)
    
    # Initialize results dictionary
    results = {}
    
    # First, get initial values (seed scores) - try to get from any method
    print("📊 Getting initial (seed) scores...")
    initial_scores = {}
    
    for peptide, name in zip(peptides, names):
        initial_score = None
        
        # Try each method to find initial score for this peptide
        for method in methods:
            folder_name = f"{method}_{name}"
            folder_path = Path(base_dir) / folder_name
            
            try:
                if folder_path.exists():
                    initial, max_scores, mean, sd = max_scores_for_sequence(folder_path)
                    if initial is not None:
                        initial_score = initial
                        print(f"  {name}: {initial_score:.3f} (from {method})")
                        break
            except:
                continue
        
        if initial_score is None:
            print(f"  {name}: No initial score found")
            
        initial_scores[name] = initial_score
    
    # Add initial scores as first row
    results["Initial (seed)"] = initial_scores
    
    # Process each optimization method
    print(f"\n🔬 Processing optimization methods...")
    
    for method in methods:
        print(f"\nProcessing {method.upper()}...")
        method_scores = {}
        
        for peptide, name in zip(peptides, names):
            folder_name = f"{method}_{name}"
            folder_path = Path(base_dir) / folder_name
            
            try:
                if folder_path.exists():
                    initial, max_scores, mean, sd = max_scores_for_sequence(folder_path)
                    
                    if max_scores and len(max_scores) > 0:
                        if len(max_scores) > 1:
                            # Multiple runs - show mean ± std
                            method_scores[name] = f"{mean:.2f} ± {sd:.2f}"
                            print(f"  {name}: {mean:.2f} ± {sd:.2f} (n={len(max_scores)})")
                        else:
                            # Single run
                            method_scores[name] = f"{mean:.2f}"
                            print(f"  {name}: {mean:.2f} (n=1)")
                    else:
                        method_scores[name] = "No data"
                        print(f"  {name}: No data")
                else:
                    method_scores[name] = "Folder not found"
                    print(f"  {name}: Folder {folder_name} not found")
                    
            except Exception as e:
                method_scores[name] = "Error"
                print(f"  {name}: Error - {str(e)[:50]}...")
        
        results[method] = method_scores
    
    # Convert to DataFrame
    df = pd.DataFrame(results).T  # Transpose so methods are rows
    
    # Reorder columns to match the names order
    df = df[names]
    
    print(f"\n{'='*60}")
    print("RESULTS TABLE")
    print(f"{'='*60}")
    print(df.to_string())
    
    # Save to CSV
    output_path = Path(base_dir) / "results_table.csv"
    df.to_csv(output_path)
    print(f"\n💾 Saved to: {output_path}")
    
    return df


def create_latex_results_table(methods, peptides, names, base_dir="csv_outputs"):
    """
    Create a LaTeX-formatted results table.
    
    Args:
        methods (list): List of optimization methods
        peptides (list): List of peptide sequences
        names (list): List of peptide names
        base_dir (str): Base directory containing CSV folders
    
    Returns:
        str: LaTeX table string
    """
    
    # First create the pandas DataFrame
    df = create_results_table(methods, peptides, names, base_dir)
    
    print(f"\n{'='*60}")
    print("GENERATING LATEX TABLE")
    print(f"{'='*60}")
    
    # Start LaTeX table
    latex_lines = []
    latex_lines.append("\\begin{table}[ht]")
    latex_lines.append("\\centering")
    latex_lines.append("\\caption{Maximal log2 MIC values achieved by each optimization method. Reported values are the mean and standard deviation over multiple runs. The top row shows the predicted log2 MIC of seeds before optimization.}")
    latex_lines.append("\\label{tab:optimization_results}")
    
    # Table setup
    n_cols = len(names) + 1  # +1 for method column
    latex_lines.append(f"\\begin{{tabular}}{{l{'c' * len(names)}}}")
    latex_lines.append("\\toprule")
    
    # Header
    header = "Method & " + " & ".join(names) + " \\\\"
    latex_lines.append(header)
    latex_lines.append("\\midrule")
    
    # Rows
    for method_name, row in df.iterrows():
        # Format method name
        if method_name == "Initial (seed)":
            formatted_method = "Initial (seed)"
        else:
            method_mapping = {
                'adalead': 'AdaLead',
                'bo': 'BO',
                'cbas': 'CbAS',
                'cmaes': 'CMA-ES',
                'dynappo': 'DynaPPO',
                'gfn_al': 'GFN-AL',
                'gfn_al_cs': 'GFN-AL-δCS',
                'pex': 'PEX'
            }
            formatted_method = method_mapping.get(method_name, method_name.upper())
        
        # Format row values
        row_values = []
        for name in names:
            value = row[name]
            if isinstance(value, str) and value not in ["No data", "Error", "Folder not found"]:
                row_values.append(value)
            elif isinstance(value, (int, float)):
                row_values.append(f"{value:.2f}")
            else:
                row_values.append("--")
        
        row_line = f"{formatted_method} & " + " & ".join(row_values) + " \\\\"
        latex_lines.append(row_line)
    
    # Close table
    latex_lines.append("\\bottomrule")
    latex_lines.append("\\end{tabular}")
    latex_lines.append("\\end{table}")
    
    latex_table = "\n".join(latex_lines)
    
    print("LaTeX Table:")
    print("-" * 40)
    print(latex_table)
    
    # Save LaTeX to file
    latex_path = Path(base_dir) / "results_table.tex"
    with open(latex_path, 'w') as f:
        f.write(latex_table)
    print(f"\n💾 LaTeX saved to: {latex_path}")
    
    return latex_table

In [12]:
# Create the comprehensive results table
print("🎯 Creating results table from CSV folders...")
print(f"Methods: {methods}")
print(f"Peptide names: {names}")
print("="*60)

results_table = create_results_table(methods, peptides, names, "csv_outputs")

print(f"\n🎉 Results table creation completed!")
print(f"📊 Table shape: {results_table.shape}")
print(f"📋 Preview:")
print(results_table)

🎯 Creating results table from CSV folders...
Methods: ['bo', 'cbas', 'adalead', 'dynappo', 'gfn_al', 'gfn_al_cs', 'cmaes', 'pex']
Peptide names: ['KY14', 'KF16', 'KK16', 'FL14', 'mammuthusin-3', 'hydrodamin-2']
🎯 Creating comprehensive results table...
📊 Getting initial (seed) scores...
Found 5 CSV files in csv_outputs\bo_KY14
  bo_KY14_seed_1.csv: max_score = 0.755
  bo_KY14_seed_10.csv: max_score = 0.723
  bo_KY14_seed_2.csv: max_score = 0.539
  bo_KY14_seed_3.csv: max_score = 0.714
  bo_KY14_seed_4.csv: max_score = 0.838

Summary: 5 files processed
Mean max score: 0.714 ± 0.098
  KY14: 1.877 (from bo)
  KF16: No initial score found
Found 5 CSV files in csv_outputs\bo_KK16
  bo_KK16_seed_1.csv: max_score = 0.622
  bo_KK16_seed_10.csv: max_score = 0.726
  bo_KK16_seed_2.csv: max_score = 0.570
  bo_KK16_seed_3.csv: max_score = 0.575
  bo_KK16_seed_4.csv: max_score = 0.539

Summary: 5 files processed
Mean max score: 0.606 ± 0.066
  KK16: 1.849 (from bo)
Found 5 CSV files in csv_outputs\